# 03 — Modelagem: as três trilhas na mesma tabela

Roda baselines, GNN relacional e GNN geográfica sobre **a mesma partição e os
mesmos exemplos**, e reporta tudo junto.

Isso não é conveniência de apresentação, é a condição para o resultado significar
alguma coisa. A versão anterior deste notebook avaliava a GNN com
`train_mask` como máscara de teste — o número reportado como desempenho de teste
era desempenho de treino — e não tinha baseline nenhuma ao lado. Ver D-11.

**Como ler os números.** A prevalência é 0,065%, um positivo a cada 1.530
candidatos. O valor absoluto do average precision não é interpretável nessa
escala: um AP de 0,02 é trinta vezes a linha de base e ainda parece zero. A
métrica de destaque é **MAP@k por estabelecimento**, que ranqueia os 99 tipos de
equipamento dentro de cada estabelecimento. Ver D-19.

In [ ]:
import sys
from pathlib import Path

BASE_DIR = Path.cwd().parent
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

import pandas as pd
import torch

from src import baselines, changes, gnn, graph, metrics, tasks
from src.splits import particionar

pd.set_option("display.width", 200)
torch.manual_seed(42)

PERIODOS = changes.periodos_disponiveis()
PARTICAO = particionar(PERIODOS)
print(f"snapshots: {PERIODOS}\n")
print(PARTICAO.resumo())
print(f"\nfim da janela de treino: {PARTICAO.fim_do_treino}")
print("Nenhuma feature de treino pode enxergar além dessa data.")

## A tarefa

Uma tabela de rótulos, consumida por todas as trilhas. Nenhuma delas recalcula a
partição — é isso que garante que comparem os mesmos exemplos.

In [ ]:
tarefa = tasks.tarefa_aquisicao(PARTICAO)
print(f"{tarefa.nome}  |  {len(tarefa.df):,} exemplos  |  prevalência {tarefa.prevalencia:.5%}")
print()
print(tarefa.resumo().to_string(index=False))

## Trilha 1 — baselines sem estrutura

Cinco modelos que não veem relação nem vizinhança. A diferença entre eles e as
trilhas 2 e 3 é a medida do valor da estrutura.

`persistencia` prevê zero por construção, já que todo candidato é um par ausente
em `t`. O AP dela é exatamente a prevalência — é a régua, não um competidor.

In [ ]:
previsoes = baselines.rodar_todas(tarefa, PARTICAO, conjunto="teste")

resultados = {
    nome: metrics.avaliar_classificacao(p.y, p.escore, p.entidades, k=10)
    for nome, p in previsoes.items()
}
print(metrics.tabela_de_resultados(resultados).to_string())
print()
for nome, p in previsoes.items():
    print(f"{nome}: {p.metadados}")

## Trilha 2 — GNN relacional

O grafo do schema CNES inteiro. Nós são estabelecimentos e tabelas de fato;
arestas são as chaves estrangeiras de `docs/01-selecao-tabelas.md`.

Montar o grafo é a parte cara: o filtro por município é empurrado para dentro do
scan Parquet, senão seria preciso carregar o país inteiro para descartar 98%.

In [ ]:
db = graph.montar_db(municipio_id=graph.MUNICIPIO_SAO_PAULO)
print(f"tabelas no grafo: {len(db.table_dict)}")
for nome, t in sorted(db.table_dict.items(), key=lambda kv: -kv[1].df.num_rows)[:8]:
    print(f"  {nome:28} {t.df.num_rows:>10,} linhas  fkeys={list((t.fkey_col_to_pkey_table or {}))}")

In [ ]:
unidades = sorted(set(db.table_dict[graph.TABELA_RAIZ].df[graph.COL_ENTIDADE].to_pylist()))
itens = sorted(tarefa.df[tarefa.col_item].dropna().unique())
indice = gnn.IndicePares.de(unidades, itens)
print(f"{len(unidades):,} estabelecimentos  x  {len(itens)} tipos de equipamento")

# As features param no fim da janela de treino: ler adiante é vazamento mesmo
# com os rótulos corretamente divididos.
features = gnn.features_de_estabelecimento(db, unidades, ate_periodo=PARTICAO.fim_do_treino)
print(f"matriz de features: {tuple(features.shape)}")

dados_rel = gnn.grafo_relacional_para_data(db, unidades, features)
print(f"tipos de nó: {len(dados_rel.node_types)}  tipos de aresta: {len(dados_rel.edge_types)}")

In [ ]:
modelo_rel, hist_rel = gnn.treinar_aquisicao(
    tarefa, PARTICAO, dados_rel, indice, epocas=200, paciencia=20, verboso=True
)
print(f"\nmelhor época {hist_rel['melhor_epoca']}  "
      f"AP validação {hist_rel['melhor_ap_validacao']:.5f}  "
      f"({hist_rel['epocas_rodadas']} épocas, {hist_rel['dispositivo']})")

prev_rel = gnn.prever_aquisicao(modelo_rel, tarefa, dados_rel, indice,
                                conjunto="teste", nome="gnn_relacional")
print(prev_rel.metadados)

## Trilha 3 — GNN geográfica

Só estabelecimentos e proximidade física. Ignora a estrutura de tabelas por
construção.

**Ressalva obrigatória (D-15, D-17).** Só 57% dos estabelecimentos de São Paulo
têm coordenada utilizável, e o teto é estrutural — quem não tem em 2022 nunca
teve. A trilha 3 fala sobre esse subconjunto, e a comparação com as outras duas
precisa ser feita **sobre os mesmos nós**, senão mede diferença de amostra em vez
de diferença de estrutura.

In [ ]:
grafo_geo = graph.montar_grafo_geografico(db, k=10)
print(f"nós posicionáveis: {grafo_geo.n_nos:,} de {len(unidades):,} "
      f"({100 * grafo_geo.n_nos / len(unidades):.1f}%)")
print(f"arestas: {grafo_geo.n_arestas:,}  (kNN k={grafo_geo.k}, simetrizado)")

indice_geo = gnn.IndicePares.de(grafo_geo.unidades, itens)
features_geo = gnn.features_de_estabelecimento(
    db, grafo_geo.unidades, ate_periodo=PARTICAO.fim_do_treino
)
dados_geo = gnn.grafo_geografico_para_data(grafo_geo, features_geo)
print(f"features: {tuple(features_geo.shape)}")

In [ ]:
modelo_geo, hist_geo = gnn.treinar_aquisicao(
    tarefa, PARTICAO, dados_geo, indice_geo, epocas=200, paciencia=20, verboso=True
)
print(f"\nmelhor época {hist_geo['melhor_epoca']}  "
      f"AP validação {hist_geo['melhor_ap_validacao']:.5f}")

prev_geo = gnn.prever_aquisicao(modelo_geo, tarefa, dados_geo, indice_geo,
                                conjunto="teste", nome="gnn_geografica")
print(prev_geo.metadados)

## Resultado

A regra de D-11: nenhuma métrica de GNN aparece sem as baselines na mesma tabela.

A primeira tabela usa todos os exemplos de teste, o que é injusto com a trilha 3 —
ela só pontua os 57% posicionáveis. A segunda restringe todas as trilhas ao mesmo
subconjunto, e é a comparação que de fato responde se a estrutura geográfica
acrescenta algo.

In [ ]:
todas = dict(previsoes)
todas["gnn_relacional"] = prev_rel
todas["gnn_geografica"] = prev_geo

completa = {
    nome: metrics.avaliar_classificacao(p.y, p.escore, p.entidades, k=10)
    for nome, p in todas.items()
}
print("=== todos os exemplos de teste ===")
print("(a trilha geográfica avalia menos pares; ver a tabela pareada abaixo)\n")
print(metrics.tabela_de_resultados(completa).to_string())

In [ ]:
# Comparação pareada: só os estabelecimentos que a trilha geográfica alcança.
import numpy as np

posicionaveis = set(grafo_geo.unidades)

def restringir(p):
    dentro = np.isin(p.entidades, list(posicionaveis))
    return metrics.avaliar_classificacao(
        p.y[dentro], p.escore[dentro], p.entidades[dentro], k=10
    )

pareada = {nome: restringir(p) for nome, p in todas.items()}
print(f"=== restrito aos {len(posicionaveis):,} estabelecimentos posicionáveis ===\n")
print(metrics.tabela_de_resultados(pareada).to_string())

### Experimento de controle: sem as transições de pandemia

As transições que tocam 2020 ou 2021 atravessam um regime de aquisição
excepcional. A metodologia (seção 4.1) exige rodar a variante sem elas e relatar
se a conclusão muda.

In [ ]:
particao_sem_covid = particionar(PERIODOS, excluir_pandemia=True)
print(particao_sem_covid.resumo())

tarefa_sem_covid = tasks.tarefa_aquisicao(particao_sem_covid)
prev_sem_covid = baselines.rodar_todas(tarefa_sem_covid, particao_sem_covid, conjunto="teste")

controle = {
    nome: metrics.avaliar_classificacao(p.y, p.escore, p.entidades, k=10)
    for nome, p in prev_sem_covid.items()
}
print()
print(metrics.tabela_de_resultados(controle).to_string())
print("\nA ordenação entre modelos se mantém, ou o choque de covid a invertia?")

## Leitura dos resultados

Preencher depois de executar. As três perguntas que a tabela precisa responder:

1. **Alguma trilha supera a persistência ingênua?** Se não, nada foi aprendido —
   e o AP da persistência é literalmente a prevalência.
2. **As GNNs superam o GBDT tabular?** Essa diferença é o valor da estrutura, que
   é a contribuição do trabalho. Se não houver diferença, a conclusão honesta é
   que a estrutura relacional do CNES não ajuda nesta tarefa — resultado negativo
   publicável, não um fracasso.
3. **A trilha geográfica supera a relacional na comparação pareada?** Responde se
   escassez se explica melhor por vizinhança física ou por estrutura
   administrativa.

Toda conclusão precisa citar a tabela **pareada**, não a completa, sempre que a
trilha geográfica estiver envolvida.